In [1]:
import torch
import timm
import pandas as pd

In [2]:
# The classification models and their timm names
backbone_models = {
    "ConvNext_Nano": "convnext_nano.in12k",
    "DeiT": "deit_base_distilled_patch16_224.fb_in1k",
    "EfficientNetV2L": "tf_efficientnetv2_l.in21k",
    "EfficientNetV2M": "efficientnetv2_rw_m.agc_in1k",
    "EfficientNetV2S": "efficientnetv2_rw_s.ra2_in1k",
    "EVA02_Large":"eva02_large_patch14_224.mim_in22k",
    "InceptionNext_Tiny": "inception_next_tiny.sail_in1k",
    "ResNet101": "resnet101.a1_in1k",
    "SeResNext101": "seresnext101_32x4d.gluon_in1k",
    "SWIN_V2": "swin_base_patch4_window7_224.ms_in1k",
}

In [3]:
results = []

for friendly_name, timm_name in backbone_models.items():
    try:
        # pretrained=False saves bandwidth/time; we only need the architecture
        model = timm.create_model(timm_name, pretrained=False)
        
        # Calculate parameters
        total_params = sum(p.numel() for p in model.parameters())
        learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        results.append({
            "model_name": friendly_name,
            "total_params": total_params,
            "learnable_params": learnable_params
        })
        
        # Clean up memory immediately
        del model        
    except Exception as e:
        print(f"Could not load {timm_name}: {e}")

# Create the DataFrame
df_params = pd.DataFrame(results)
df_params

,model_name,total_params,learnable_params
0,ConvNext_Nano,22529821,22529821
1,DeiT,87338192,87338192
2,EfficientNetV2L,145215155,145215155
3,EfficientNetV2M,53236442,53236442
4,EfficientNetV2S,23941296,23941296
5,EVA02_Large,303268800,303268800
6,InceptionNext_Tiny,28055680,28055680
7,ResNet101,44549160,44549160
8,SeResNext101,48955416,48955416
9,SWIN_V2,87768224,87768224


In [4]:
df_params['t_params_in_mil'] = df_params['total_params'] / 1e6
df_params['l_params_in_mil'] = df_params['learnable_params'] / 1e6
df_params

,model_name,total_params,learnable_params,t_params_in_mil,l_params_in_mil
0,ConvNext_Nano,22529821,22529821,22.529821,22.529821
1,DeiT,87338192,87338192,87.338192,87.338192
2,EfficientNetV2L,145215155,145215155,145.215155,145.215155
3,EfficientNetV2M,53236442,53236442,53.236442,53.236442
4,EfficientNetV2S,23941296,23941296,23.941296,23.941296
5,EVA02_Large,303268800,303268800,303.268800,303.268800
6,InceptionNext_Tiny,28055680,28055680,28.055680,28.055680
7,ResNet101,44549160,44549160,44.549160,44.549160
8,SeResNext101,48955416,48955416,48.955416,48.955416
9,SWIN_V2,87768224,87768224,87.768224,87.768224


Now count the parameters of PADC models

In [5]:
from tqdm import tqdm
from torch import nn
import segmentation_models_pytorch as smp

# Hyperparameters and paths
DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
NUM_PARTS = 4
CLASSIFIER_NAME = "deit_base_distilled_patch16_224.fb_in1k"
SEGMENTATION_NAME = "deeplabv3plus"
SEGMENTATION_ENCODER = "resnext50_32x4d"
SEGMENTATION_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/2_segmentation/Beemachine/deeplabv3+/lightning_logs/version_0/checkpoints/epoch=199-step=54400.ckpt"
FEATURE_SIZE = 937
IMAGE_SIZE = 224
DEVICE_ID = 0

In [6]:
# Load segmentation model
def load_segmentation(name, encoder_name, checkpoint_path):    
    # Define the segmentation model and load weights
    model = smp.create_model(
                name,
                encoder_name=encoder_name,
                in_channels = 3, # 3 for RGB images
                classes = NUM_PARTS
            ).to(f"cuda:{DEVICE_ID}")
    # checkpoint_path = r"/home/c/choton/beemachine/codes/everyday_test/oct1_25/segmentation_models/lightning_logs/fpn/lightning_logs/version_0/checkpoints/epoch=199-step=37400.ckpt"
    checkpoint = torch.load(checkpoint_path, map_location=rf"cuda:{DEVICE_ID}")

    # Sometimes Lightning adds "state_dict"
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

        # Remove "model." or "net." prefixes if present
        new_state_dict = {k.replace("model.", "").replace("net.", ""): v 
                        for k, v in state_dict.items() if k not in ["mean", "std"]}

        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(checkpoint)
    # model.to(f"cuda:{DEVICE_ID}")
    return model

def extract_all_features(image_np, mask_np):
    # Return dummy shapes
    record = {f'part_{i}': 0 for i in range(FEATURE_SIZE)}
    return record

In [7]:
class ShapeEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, embed_dim)

    def forward(self, mask_tensor):
        """
        mask_tensor: (B, 3, 64, 64)
        """
        feat = self.encoder(mask_tensor)
        feat = feat.flatten(1)
        z = self.fc(feat)
        return z  

class GatedFusion(nn.Module):
    def __init__(self, img_dim, shape_dim):
        super().__init__()

        self.shape_proj = nn.Linear(shape_dim, img_dim)

        self.gate = nn.Sequential(
            nn.Linear(img_dim + img_dim, img_dim),
            nn.LayerNorm(img_dim),
            nn.Sigmoid()
        )

    def forward(self, z_img, z_shape):
        z_shape_proj = self.shape_proj(z_shape)

        concat = torch.cat([z_img, z_shape_proj], dim=1)
        g = self.gate(concat)

        fused = z_img + g * (z_shape_proj - z_img)
        return fused

class DeepShapeFusionModel(nn.Module):
    def __init__(self, num_classes, classifier_name, shape_embed_dim=256):
        super().__init__()
        self.num_shape_feats = FEATURE_SIZE

        # Modules
        self.seg_module = load_segmentation(
            name=SEGMENTATION_NAME, 
            encoder_name=SEGMENTATION_ENCODER, 
            checkpoint_path=SEGMENTATION_WEIGHTS
        )
        self.backbone = timm.create_model(model_name=classifier_name, pretrained=True, num_classes=0)
        self.shape_encoder = ShapeEncoder(shape_embed_dim)
        self.feature_dim = self.backbone.num_features

        self.fusion = GatedFusion(
            img_dim=self.backbone.num_features,
            shape_dim=shape_embed_dim
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        total_feats = self.backbone.num_features + self.num_shape_feats # + shape_embed_dim

        self.classifier = nn.Sequential(
                                nn.LayerNorm(total_feats),
                                nn.Dropout(0.3),
                                nn.Linear(total_feats, num_classes)
                            )

        for p in self.seg_module.parameters():
            p.requires_grad = False
        self.seg_module.eval()

    def forward(self, x, shape_feats=None):

        # ---- Segmentation ----
        with torch.no_grad():
            mask_logits = self.seg_module(x)
            masks = torch.argmax(mask_logits, dim=1)  # (B, H, W) with labels 0,1,2,3
        # mask_logits = self.seg_module(x)
        # mask_probs = torch.softmax(mask_logits, dim=1)  # example single part

        # Extract part embeddings
        parts = mask_logits[:, 1:, :, :]  # remove background, will try mask_probs as well
        mask_binary = parts.sum(dim=1, keepdim=True)
        edge = torch.abs(
            torch.nn.functional.avg_pool2d(mask_binary, 3, stride=1, padding=1)
            - mask_binary
        )
        shape_tensor = torch.cat([mask_binary, edge, mask_binary], dim=1)
        z_shape = self.shape_encoder(shape_tensor)

        # ---- Extract hand-crafted shape descriptors ----
        # ---- Shape features ----
        if shape_feats is None:
            batch_size = x.size(0)
            batch_feats = []
            for i in range(batch_size):
                img_np = x[i].detach().cpu().numpy().transpose(1, 2, 0)
                mask_np = masks[i].detach().cpu().numpy()
                feats_dict = extract_all_features(img_np, mask_np)
                # Convert dict to tensor (relies on consistent insertion order for 937 features)
                feat_tensor = torch.tensor(list(feats_dict.values()), dtype=torch.float32)
                batch_feats.append(feat_tensor)
            shape_feats = torch.stack(batch_feats).to(x.device)  # (B, 937)
        # shape_feats = torch.tensor(shape_feats).to(x.device)        

        # ---- Image Encoding ----
        z_img = self.backbone.forward_features(x)

        # Make layout agnostic: if NHWC (common for Swin), permute to NCHW
        if z_img.dim() == 4 and z_img.shape[1] != self.feature_dim:
            # Assume NHWC if channels are in last dim
            z_img = z_img.permute(0, 3, 1, 2)  # → [B, C, H, W]
        
        if z_img.dim() == 3:  # Token-based (ViT/DeiT style)
            # For distilled DeiT: cls token at index 0, distillation token at index 1
            # → Use cls token (most common) or average cls + distill for better performance
            # z_img = z_img[:, 0]  # [B, 768] – cls token (recommended for classification tasks)

            # Alternative (often slightly better for distilled models):
            z_img = z_img[:, :2].mean(dim=1)  # average cls + distill token → [B, 768]

        else:  # 4D spatial feature map (conv or Swin-style)
            # Ensure NCHW layout
            if z_img.dim() == 4 and z_img.shape[1] != self.feature_dim:
                z_img = z_img.permute(0, 3, 1, 2)  # NHWC → NCHW

            # Now safe to pool (expects NCHW)      
            # z_img = z_img.mean(-1).mean(-1)
            z_img = self.global_pool(z_img) # → [B, C, 1, 1]
            z_img = z_img.flatten(start_dim=1) # → [B, C]
        # print("Shape of extracted features from visual,", z_img.shape)
        fused = self.fusion(z_img, z_shape)
        combined = torch.cat([fused, shape_feats], dim=1)  # (B, total_feats)

        # ---- Classification ----
        out = self.classifier(combined)

        return out, mask_logits, z_shape


In [8]:
device = torch.device(f"cuda:{DEVICE_ID}")
model = DeepShapeFusionModel(num_classes=160, classifier_name=CLASSIFIER_NAME, shape_embed_dim=512)
model.to(device)
# model.load_state_dict(state_dict)
model.eval()

DeepShapeFusionModel(
  (seg_module): DeepLabV3Plus(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_runni

In [9]:
# Calculate parameters
total_params = sum(p.numel() for p in model.parameters())
learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params/1e6, learnable_params/1e6

(113.96247, 87.812242)

Now check for all backbones

In [10]:
results = []

for friendly_name, timm_name in tqdm(backbone_models.items(), desc="Calculating Model Parameters"):
    print("Loading backbone:", friendly_name)
    try:
        # pretrained=False saves bandwidth/time; we only need the architecture
        model_baseline = timm.create_model(timm_name, pretrained=False, num_classes=160)
        model_PADC = DeepShapeFusionModel(num_classes=160, classifier_name=timm_name, shape_embed_dim=512)
        
        # Calculate parameters
        total_base_params = sum(p.numel() for p in model_baseline.parameters())
        learnable_base_params = sum(p.numel() for p in model_baseline.parameters() if p.requires_grad)
        total_padc_params = sum(p.numel() for p in model_PADC.parameters())
        learnable_padc_params = sum(p.numel() for p in model_PADC.parameters() if p.requires_grad)
        
        results.append({
            "model_name": friendly_name,
            "total_base_params": total_base_params,
            "learnable_base_params": learnable_base_params,
            "total_PADC_params": total_padc_params,
            "learnable_PADC_params": learnable_padc_params,
        })
        
        # Clean up memory immediately
        del model_baseline, model_PADC        
    except Exception as e:
        print(f"Could not load {timm_name}: {e}")

# Create the DataFrame
df_params = pd.DataFrame(results)
df_params

Calculating Model Parameters:   0%|          | 0/10 [00:00<?, ?it/s]

Loading backbone: ConvNext_Nano


Calculating Model Parameters:  10%|█         | 1/10 [00:03<00:34,  3.88s/it]

Loading backbone: DeiT


Calculating Model Parameters:  20%|██        | 2/10 [00:11<00:46,  5.86s/it]

...

Loading backbone: SeResNext101


Calculating Model Parameters:  90%|█████████ | 9/10 [00:58<00:05,  5.78s/it]

Loading backbone: SWIN_V2


Calculating Model Parameters: 100%|██████████| 10/10 [01:09<00:00,  6.92s/it]


,model_name,total_base_params,learnable_base_params,total_PADC_params,learnable_PADC_params
0,ConvNext_Nano,15055120,15055120,42667606,16517378
1,DeiT,86046272,86046272,113962470,87812242
2,EfficientNetV2L,117439232,117439232,147840838,121690610
3,EfficientNetV2M,51427922,51427922,88266632,62116404
4,EfficientNetV2S,22435176,22435176,56247726,30097498
5,EVA02_Large,303432800,303432800,332522150,306371922
6,InceptionNext_Tiny,26119480,26119480,53912958,27762730
7,ResNet101,42828000,42828000,78739238,52589010
8,SeResNext101,47234256,47234256,83145494,56995266
9,SWIN_V2,86907224,86907224,115996574,89846346


In [11]:
df_mil = df_params.drop(columns='model_name')
df_mil = df_mil / 1e6
df_mil['model_name'] = df_params['model_name']
df_new_mil = df_mil[['model_name', 'learnable_base_params', 'learnable_PADC_params']]
df_new_mil

,model_name,learnable_base_params,learnable_PADC_params
0,ConvNext_Nano,15.055120,16.517378
1,DeiT,86.046272,87.812242
2,EfficientNetV2L,117.439232,121.690610
3,EfficientNetV2M,51.427922,62.116404
4,EfficientNetV2S,22.435176,30.097498
5,EVA02_Large,303.432800,306.371922
6,InceptionNext_Tiny,26.119480,27.762730
7,ResNet101,42.828000,52.589010
8,SeResNext101,47.234256,56.995266
9,SWIN_V2,86.907224,89.846346


In [12]:
x = df_new_mil.to_latex(index=False)
print(x)

\begin{tabular}{lrr}
\toprule
model_name & learnable_base_params & learnable_PADC_params \\
\midrule
ConvNext_Nano & 15.055120 & 16.517378 \\
DeiT & 86.046272 & 87.812242 \\
EfficientNetV2L & 117.439232 & 121.690610 \\
EfficientNetV2M & 51.427922 & 62.116404 \\
EfficientNetV2S & 22.435176 & 30.097498 \\
EVA02_Large & 303.432800 & 306.371922 \\
InceptionNext_Tiny & 26.119480 & 27.762730 \\
ResNet101 & 42.828000 & 52.589010 \\
SeResNext101 & 47.234256 & 56.995266 \\
SWIN_V2 & 86.907224 & 89.846346 \\
\bottomrule
\end{tabular}

